# Specialty Prioritisation Framework 
### Purpose:

Define framework
- Import indicator datasets
- Build specialty master datasets
- Run EDA
- Export outputs

Develop a reproducible framework to identify clinical specialties demonstrating strong signals of:
 
- Disease burden
- Poor outcomes
- System pressure
- Geographic variation
 
##### This framework is intended to support Horizon Scanning and Demand Signalling by providing a transparent evidence base for identifying themes that may warrant further investigation.

## Set-up and Parameters

In [1]:
#import relevant libraries

from pathlib import Path
import pandas as pd
import numpy as np
import seaborn as sns
from functools import reduce

INPUT_DIR = Path("../Inputs")
OUTPUT_DIR = Path("../outputs")
CONFIG_FILE = INPUT_DIR / "indicator_config.xlsx"
POPULATION_FILE = INPUT_DIR / "ICB_ONS_2024.xlsx"
IMD_FILE = INPUT_DIR / "ICB_IMD_2025.xlsx"
ICB_LOOKUP_FILE = INPUT_DIR / "icb_lookup.xlsx"

EXPECTED_ICBS = 42
# to flag in the coverage report if not all ICBs have data
COVERAGE_THRESHOLD = 100

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

framework = pd.read_excel(CONFIG_FILE)

framework["Include"] = (
    framework["Include"]
    .astype(str)
    .str.strip()
    .str.lower()
    .eq("yes")
)

active_config = framework.loc[framework["Include"]].copy()

display(active_config)

,Include,Specialty,Domain,Indicator,File_Name,Source,Direction,Status
0,True,Cardiovascular,Outcome,Under75_CVD_Mortality,Under75_CVD_Mortality.xlsx,MHS,Higher_Worse,Included in POC
1,True,Cardiovascular,Pressure,Cardiology_RTT_18plus,Cardiology_RTT_18plus.xlsx,MHS,Higher_Worse,Included in POC
2,True,Cardiovascular,Pressure,Cardiology_ElectiveAdmissions_20day+,Cardiology_ElectiveAdmissions_20day+.xlsx,MHS,Higher_Worse,Included in POC
3,True,Cardiovascular,Pressure,Cardiology_EmergencyAdmissions_20day+,Cardiology_EmergencyAdmissions_20day+.xlsx,MHS,Higher_Worse,Included in POC
4,True,Cardiovascular,Burden,Heart_Failure_Prevalence,Heart_Failure_Prevalence.xlsx,MHS,Higher_Worse,Included in POC
5,True,Cardiovascular,Burden,Hypertension_Prevalence,Hypertension_Prevalence.xlsx,MHS,Higher_Worse,Included in POC
6,True,Cardiovascular,Burden,CoronaryHeartDisease_Prevalence,CoronaryHeartDisease_Prevalence.xlsx,MHS,Higher_Worse,Included in POC
7,True,Cardiovascular,Burden,PeripheralArterialDisease_Prevalence,PeripheralArterialDisease_Prevalence.xlsx,MHS,Higher_Worse,Included in POC
8,True,Respiratory,Outcome,Under75_Respiratory_Mortality,Under75_Respiratory_Mortality.xlsx,MHS,Higher_Worse,Included in POC
9,True,Respiratory,Pressure,Respiratory_RTT_18plus,Respiratory_RTT_18plus.xlsx,MHS,Higher_Worse,Included in POC


## Load Configuration

In [2]:
def load_mhs_indicator(file_path, indicator_name):
    required_columns = [
        "OrganisationName",
        "OrganisationCode",
        "OrganisationRawValue"
    ]

    df = pd.read_excel(file_path)

    missing_columns = [
        column for column in required_columns
        if column not in df.columns
    ]

    if missing_columns:
        raise ValueError(
            f"{file_path.name} is missing columns: {missing_columns}"
        )

    df = df[required_columns].copy()

    df = df.rename(columns={
        "OrganisationName": "ICB_Name",
        "OrganisationCode": "ICB_Code",
        "OrganisationRawValue": indicator_name
    })

    df["ICB_Name"] = df["ICB_Name"].astype(str).str.strip()
    df["ICB_Code"] = df["ICB_Code"].astype(str).str.strip()

    raw_values = df[indicator_name].copy()

    df[indicator_name] = pd.to_numeric(
        df[indicator_name],
        errors="coerce"
    )

    non_numeric_mask = (
        raw_values.notna()
        & df[indicator_name].isna()
    )

    if non_numeric_mask.any():
        non_numeric_values = sorted(
            raw_values.loc[non_numeric_mask]
            .astype(str)
            .unique()
        )

        print(
            f"WARNING: {file_path.name} contains "
            f"non-numeric values converted to missing: "
            f"{non_numeric_values}"
        )

    duplicate_codes = df["ICB_Code"].duplicated(keep=False)

    if duplicate_codes.any():
        duplicate_values = sorted(
            df.loc[duplicate_codes, "ICB_Code"].unique()
        )

        raise ValueError(
            f"{file_path.name} contains duplicate ICB codes: "
            f"{duplicate_values}"
        )

    return df

## Add configuration validation before loading indicators

In [3]:
REQUIRED_CONFIG_COLUMNS = [
    "Include",
    "Specialty",
    "Domain",
    "Indicator",
    "File_Name",
    "Source",
    "Direction",
    "Status"
]

missing_config_columns = [
    column for column in REQUIRED_CONFIG_COLUMNS
    if column not in framework.columns
]

if missing_config_columns:
    raise ValueError(
        "The configuration file is missing required columns: "
        f"{missing_config_columns}"
    )

In [4]:
CONFIG_TEXT_COLUMNS = [
    "Specialty",
    "Domain",
    "Indicator",
    "File_Name",
    "Source",
    "Direction",
    "Status"
]

for column in CONFIG_TEXT_COLUMNS:
    framework[column] = (
        framework[column]
        .astype("string")
        .str.strip()
    )

In [5]:
#check for missing values and duplicates 

active_config = framework.loc[
    framework["Include"]
].copy()

missing_config_values = active_config[
    REQUIRED_CONFIG_COLUMNS
].isna().any(axis=1)

if missing_config_values.any():
    raise ValueError(
        "Some included configuration rows contain missing values:\n"
        f"{active_config.loc[missing_config_values]}"
    )

duplicate_indicators = active_config[
    active_config["Indicator"].duplicated(keep=False)
]

if not duplicate_indicators.empty:
    raise ValueError(
        "Indicator names must be unique. Duplicates found:\n"
        f"{duplicate_indicators[['Specialty', 'Indicator']]}"
    )

duplicate_files = active_config[
    active_config["File_Name"].duplicated(keep=False)
]

if not duplicate_files.empty:
    print(
        "WARNING: Some files are used more than once:\n",
        duplicate_files[
            ["Specialty", "Indicator", "File_Name"]
        ]
    )

specialty_inventory = (
    active_config
    .groupby("Specialty", as_index=False)
    .agg(Number_of_Indicators=("Indicator", "nunique"))
)

display(specialty_inventory)

,Specialty,Number_of_Indicators
0,Cardiovascular,8
1,Respiratory,6


## Load Indicators Automatically

In [6]:
indicator_datasets = {}

for row in active_config.itertuples(index=False):
    file_path = INPUT_DIR / row.File_Name

    if not file_path.exists():
        print(
            f"WARNING: File not found for {row.Indicator}: "
            f"{file_path}"
        )
        continue

    indicator_datasets[row.Indicator] = load_mhs_indicator(
        file_path=file_path,
        indicator_name=row.Indicator
    )

print(
    f"Loaded {len(indicator_datasets)} "
    f"of {len(active_config)} configured indicators"
)

Loaded 14 of 14 configured indicators


## Generate a Coverage Summary - to spot missing ICBs

In [7]:
## Load ICB Lookup

icb_lookup = pd.read_excel(
    ICB_LOOKUP_FILE
)

icb_lookup = (
    icb_lookup[
        ["ICB_Code", "ICB_Name"]
    ]
    .dropna(subset=["ICB_Code", "ICB_Name"])
    .assign(
        ICB_Code=lambda x: x["ICB_Code"].astype(str).str.strip(),
        ICB_Name=lambda x: x["ICB_Name"].astype(str).str.strip()
    )
    .drop_duplicates(subset="ICB_Code")
    .sort_values("ICB_Name")
    .reset_index(drop=True)
)

In [8]:
def create_coverage_summary(
    datasets,
    config,
    icb_lookup
):
    records = []

    expected_codes = set(icb_lookup["ICB_Code"])

    for indicator_name, df in datasets.items():
        config_row = config.loc[
            config["Indicator"].eq(indicator_name)
        ].iloc[0]

        available_codes = set(df["ICB_Code"])

        matched_codes = expected_codes & available_codes
        missing_codes = expected_codes - available_codes
        unexpected_codes = available_codes - expected_codes

        expected_icbs = len(expected_codes)
        matched_icbs = len(matched_codes)

        coverage_percent = round(
            matched_icbs / expected_icbs * 100,
            1
        )

        records.append({
            "Specialty": config_row["Specialty"],
            "Domain": config_row["Domain"],
            "Indicator": indicator_name,
            "Expected_ICBs": expected_icbs,
            "Rows_in_Source": len(df),
            "Unique_Codes_in_Source": len(available_codes),
            "Matched_ICBs": matched_icbs,
            "Missing_ICBs": len(missing_codes),
            "Unexpected_Codes": len(unexpected_codes),
            "Coverage_Percent": coverage_percent,
            "Coverage_Flag": (
                "Good"
                if (
                    coverage_percent >= COVERAGE_THRESHOLD
                    and len(unexpected_codes) == 0
                )
                else "Review"
            )
        })

    return (
        pd.DataFrame(records)
        .sort_values(
            ["Specialty", "Coverage_Percent"],
            ascending=[True, False]
        )
        .reset_index(drop=True)
    )

In [9]:
# Prepare population and inequalities contextual data

def find_column(
    dataframe,
    exact_names=None,
    contains=None
):
    """
    Find a source column using exact possible names or
    words contained within the column heading.
    """

    exact_names = exact_names or []
    contains = contains or []

    cleaned_columns = {
        str(column).strip().lower(): column
        for column in dataframe.columns
    }

    for expected_name in exact_names:
        matched_column = cleaned_columns.get(
            expected_name.strip().lower()
        )

        if matched_column is not None:
            return matched_column

    for column in dataframe.columns:
        cleaned_column = str(column).strip().lower()

        if all(
            search_term.lower() in cleaned_column
            for search_term in contains
        ):
            return column

    raise KeyError(
        "Could not identify the required column. "
        f"Available columns are: {list(dataframe.columns)}"
    )


In [10]:
# Population preparation function

def prepare_icb_population(
    file_path,
    icb_lookup
):
    """
    Read the ONS Mid-2024 ICB workbook and aggregate
    SICBL-level population values to 2024 ICB level.

    Retains:
        - ICB geography code, such as E54000011
        - ONS ICB name
        - Total 2024 population

    Note:
        The icb_lookup parameter is retained so the function
        call remains consistent, but matching to the Q-code
        lookup occurs later using normalised ICB names.
    """

    population_raw = pd.read_excel(
        file_path,
        sheet_name="Mid-2024 ICB 2024"
    )

    population_raw = (
        population_raw
        .dropna(how="all")
        .copy()
    )

    population_raw.columns = (
        population_raw.columns
        .astype(str)
        .str.strip()
    )

    # Identify the ICB name column

    icb_name_column = find_column(
        population_raw,
        exact_names=[
            "ICB 2024 name",
            "ICB Name 2024",
            "ICB 2024 Name",
            "Integrated Care Board Name (2024)"
        ],
        contains=[
            "icb",
            "2024",
            "name"
        ]
    )

    # Identify the ICB geography code column.
    # This is expected to contain codes such as E54000011.

    icb_geography_code_column = find_column(
        population_raw,
        exact_names=[
            "ICB 2024 code",
            "ICB Code 2024",
            "ICB 2024 Code",
            "Integrated Care Board Code (2024)"
        ],
        contains=[
            "icb",
            "2024",
            "code"
        ]
    )

    # Identify the total population column

    population_column = find_column(
        population_raw,
        exact_names=[
            "All ages",
            "All Ages",
            "Total population",
            "Total Population",
            "Population",
            "Persons",
            "Total"
        ],
        contains=[
            "total"
        ]
    )

    print(
        f"Population column selected: "
        f"{population_column}"
    )

    population = population_raw[
        [
            icb_geography_code_column,
            icb_name_column,
            population_column
        ]
    ].copy()

    population = population.rename(
        columns={
            icb_geography_code_column:
                "ICB_Geography_Code",
            icb_name_column:
                "ONS_ICB_Name",
            population_column:
                "Population_2024"
        }
    )

    # Clean geographic code and ICB name

    population["ICB_Geography_Code"] = (
        population["ICB_Geography_Code"]
        .astype("string")
        .str.strip()
        .str.upper()
    )

    population["ONS_ICB_Name"] = (
        population["ONS_ICB_Name"]
        .astype("string")
        .str.strip()
    )

    # Convert population values to numeric

    raw_population_values = (
        population["Population_2024"]
        .copy()
    )

    population["Population_2024"] = (
        pd.to_numeric(
            population["Population_2024"],
            errors="coerce"
        )
    )

    non_numeric_population = (
        raw_population_values.notna()
        & population["Population_2024"].isna()
    )

    if non_numeric_population.any():

        non_numeric_values = (
            raw_population_values.loc[
                non_numeric_population
            ]
            .astype(str)
            .unique()
            .tolist()
        )

        print(
            "WARNING: Non-numeric population values "
            "were converted to missing: "
            f"{non_numeric_values}"
        )

    population = population.dropna(
        subset=[
            "ICB_Geography_Code",
            "ONS_ICB_Name",
            "Population_2024"
        ]
    )

    # Keep only valid-looking English geography codes

    invalid_geography_codes = population.loc[
        ~population[
            "ICB_Geography_Code"
        ].str.match(
            r"^E\d+$",
            na=False
        ),
        "ICB_Geography_Code"
    ].unique()

    if len(invalid_geography_codes) > 0:

        print(
            "WARNING: Unexpected ICB geography code "
            "formats found: "
            f"{sorted(invalid_geography_codes)}"
        )

    # Aggregate the SICBL-level rows to one row per ICB

    population_icb = (
        population
        .groupby(
            [
                "ICB_Geography_Code",
                "ONS_ICB_Name"
            ],
            as_index=False
        )
        .agg(
            Population_2024=(
                "Population_2024",
                "sum"
            )
        )
    )

    # Validate geography-code uniqueness

    duplicate_geography_codes = (
        population_icb[
            "ICB_Geography_Code"
        ].duplicated(
            keep=False
        )
    )

    if duplicate_geography_codes.any():

        duplicates = (
            population_icb.loc[
                duplicate_geography_codes,
                [
                    "ICB_Geography_Code",
                    "ONS_ICB_Name"
                ]
            ]
        )

        raise ValueError(
            "Duplicate ICB geography codes remain "
            "after population aggregation:\n"
            f"{duplicates}"
        )

    # Validate expected number of ICBs

    if len(population_icb) != EXPECTED_ICBS:

        print(
            f"WARNING: Expected {EXPECTED_ICBS} "
            "aggregated ICB rows, but found "
            f"{len(population_icb)}."
        )

    print(
        f"Population data prepared: "
        f"{len(population_raw)} source rows reduced "
        f"to {len(population_icb)} ICB rows."
    )

    print(
        f"Unique ICB geography codes retained: "
        f"{population_icb['ICB_Geography_Code'].nunique()}"
    )

    return population_icb

In [11]:
# IMD preparation function

def prepare_icb_imd(
    file_path
):
    """
    Read the ICB-level IMD average score.
    """

    imd_raw = pd.read_excel(
        file_path,
        sheet_name="IMD"
    )

    imd_raw.columns = (
        imd_raw.columns
        .astype(str)
        .str.strip()
    )

    imd_code_column = find_column(
        imd_raw,
        exact_names=[
            "Integrated Care Board Code (2024)"
        ],
        contains=[
            "integrated care board",
            "code",
            "2024"
        ]
    )

    imd_name_column = find_column(
        imd_raw,
        exact_names=[
            "Integrated Care Board Name (2024)"
        ],
        contains=[
            "integrated care board",
            "name",
            "2024"
        ]
    )

    imd_score_column = find_column(
        imd_raw,
        exact_names=[
            "IMD - Average score",
            "IMD - average score"
        ],
        contains=[
            "imd",
            "average",
            "score"
        ]
    )

    imd = imd_raw[
        [
            imd_code_column,
            imd_name_column,
            imd_score_column
        ]
    ].copy()

    imd = imd.rename(
        columns={
            imd_code_column: "ICB_Code",
            imd_name_column: "IMD_ICB_Name",
            imd_score_column: "IMD_Average_Score"
        }
    )

    imd["ICB_Code"] = (
        imd["ICB_Code"]
        .astype("string")
        .str.strip()
    )

    imd["IMD_ICB_Name"] = (
        imd["IMD_ICB_Name"]
        .astype("string")
        .str.strip()
    )

    imd["IMD_Average_Score"] = (
        pd.to_numeric(
            imd["IMD_Average_Score"],
            errors="coerce"
        )
    )

    imd = imd.dropna(
        subset=[
            "ICB_Code",
            "IMD_Average_Score"
        ]
    )

    if imd["ICB_Code"].duplicated().any():
        duplicate_codes = (
            imd.loc[
                imd["ICB_Code"].duplicated(
                    keep=False
                ),
                "ICB_Code"
            ]
            .tolist()
        )

        raise ValueError(
            "Duplicate ICB codes found in the IMD data: "
            f"{duplicate_codes}"
        )

    print(
        f"IMD data prepared: "
        f"{len(imd)} ICB rows."
    )

    return imd

In [12]:
icb_population = prepare_icb_population(
    file_path=POPULATION_FILE,
    icb_lookup=icb_lookup
)

icb_imd = prepare_icb_imd(
    file_path=IMD_FILE
)

Population column selected: Total
Population data prepared: 107 source rows reduced to 42 ICB rows.
Unique ICB geography codes retained: 42
IMD data prepared: 42 ICB rows.


In [13]:
display(
    icb_population.head()
)

print(
    icb_population.columns.tolist()
)

print(
    "Rows:",
    len(icb_population)
)

print(
    "Unique geography codes:",
    icb_population[
        "ICB_Geography_Code"
    ].nunique()
)

,ICB_Geography_Code,ONS_ICB_Name,Population_2024
0,E54000008,NHS Cheshire and Merseyside Integrated Care Board,2615425.0
1,E54000010,NHS Staffordshire and Stoke-on-Trent Integrate...,1177578.0
2,E54000011,"NHS Shropshire, Telford and Wrekin Integrated ...",528407.0
3,E54000013,NHS Lincolnshire Integrated Care Board,789502.0
4,E54000015,"NHS Leicester, Leicestershire and Rutland Inte...",1175364.0


['ICB_Geography_Code', 'ONS_ICB_Name', 'Population_2024']
Rows: 42
Unique geography codes: 42


In [14]:
display(
    icb_population.head()
)

display(
    icb_imd.head()
)

,ICB_Geography_Code,ONS_ICB_Name,Population_2024
0,E54000008,NHS Cheshire and Merseyside Integrated Care Board,2615425.0
1,E54000010,NHS Staffordshire and Stoke-on-Trent Integrate...,1177578.0
2,E54000011,"NHS Shropshire, Telford and Wrekin Integrated ...",528407.0
3,E54000013,NHS Lincolnshire Integrated Care Board,789502.0
4,E54000015,"NHS Leicester, Leicestershire and Rutland Inte...",1175364.0


,ICB_Code,IMD_ICB_Name,IMD_Average_Score
0,E54000008,NHS Cheshire and Merseyside Integrated Care Board,25.531
1,E54000010,NHS Staffordshire and Stoke-on-Trent Integrate...,20.392
2,E54000011,"NHS Shropshire, Telford and Wrekin Integrated ...",20.099
3,E54000013,NHS Lincolnshire Integrated Care Board,22.239
4,E54000015,"NHS Leicester, Leicestershire and Rutland Inte...",19.793


In [15]:
## Build ICB contextual dataset using normalised ICB names

def normalise_icb_name(series):
    """
    Create a standardised ICB name for matching only.

    The final displayed ICB name and Q-code continue
    to come from icb_lookup.xlsx.
    """

    return (
        series
        .astype("string")
        .str.lower()
        .str.strip()
        .str.replace(
            "nhs ",
            "",
            regex=False
        )
        .str.replace(
            " integrated care board",
            "",
            regex=False
        )
        .str.replace(
            " icb",
            "",
            regex=False
        )
        .str.replace(
            "&",
            "and",
            regex=False
        )
        .str.replace(
            "-",
            " ",
            regex=False
        )
        .str.replace(
            ",",
            "",
            regex=False
        )
        .str.replace(
            r"\s+",
            " ",
            regex=True
        )
        .str.strip()
    )


# Create matching fields in copies of all three datasets

lookup_context = icb_lookup.copy()

lookup_context["ICB_Name_Match"] = (
    normalise_icb_name(
        lookup_context["ICB_Name"]
    )
)


population_context = icb_population.copy()

population_context["ICB_Name_Match"] = (
    normalise_icb_name(
        population_context["ONS_ICB_Name"]
    )
)


imd_context = icb_imd.copy()

imd_context["ICB_Name_Match"] = (
    normalise_icb_name(
        imd_context["IMD_ICB_Name"]
    )
)


# Check that name normalisation has not created duplicates

for dataset_name, dataframe in {
    "ICB lookup": lookup_context,
    "Population": population_context,
    "IMD": imd_context
}.items():

    duplicate_names = (
        dataframe["ICB_Name_Match"]
        .duplicated(keep=False)
    )

    if duplicate_names.any():

        duplicates = (
            dataframe.loc[
                duplicate_names,
                "ICB_Name_Match"
            ]
            .drop_duplicates()
            .tolist()
        )

        raise ValueError(
            f"{dataset_name} contains duplicate "
            f"normalised ICB names: {duplicates}"
        )


# Merge population and IMD using normalised ICB names

icb_context = (
    lookup_context
    .merge(
        population_context[
            [
                "ICB_Name_Match",
                "ICB_Geography_Code",
                "Population_2024"
            ]
        ],
        on="ICB_Name_Match",
        how="left",
        validate="one_to_one",
        indicator="Population_Merge"
    )
    .merge(
        imd_context[
            [
                "ICB_Name_Match",
                "IMD_Average_Score"
            ]
        ],
        on="ICB_Name_Match",
        how="left",
        validate="one_to_one",
        indicator="IMD_Merge"
    )
)


# Report any unmatched population records

unmatched_population = icb_context.loc[
    icb_context["Population_Merge"].ne("both"),
    [
        "ICB_Code",
        "ICB_Name",
        "ICB_Name_Match"
    ]
]

if not unmatched_population.empty:

    print(
        "WARNING: The following lookup ICBs did not "
        "match the population data:"
    )

    display(
        unmatched_population
    )


# Report any unmatched IMD records

unmatched_imd = icb_context.loc[
    icb_context["IMD_Merge"].ne("both"),
    [
        "ICB_Code",
        "ICB_Name",
        "ICB_Name_Match"
    ]
]

if not unmatched_imd.empty:

    print(
        "WARNING: The following lookup ICBs did not "
        "match the IMD data:"
    )

    display(
        unmatched_imd
    )


# Keep final fields in the required order

icb_context = icb_context[
    [
        "ICB_Geography_Code",
        "ICB_Code",
        "ICB_Name",
        "Population_2024",
        "IMD_Average_Score"
    ]
].copy()


# Validate contextual dataset

if len(icb_context) != EXPECTED_ICBS:

    raise ValueError(
        f"Expected {EXPECTED_ICBS} rows in the "
        f"ICB contextual dataset, but found "
        f"{len(icb_context)}."
    )


missing_geography_codes = icb_context.loc[
    icb_context[
        "ICB_Geography_Code"
    ].isna(),
    [
        "ICB_Code",
        "ICB_Name"
    ]
]

if not missing_geography_codes.empty:

    print(
        "WARNING: Geography codes are missing "
        "for the following ICBs:"
    )

    display(
        missing_geography_codes
    )


duplicate_geography_codes = (
    icb_context[
        "ICB_Geography_Code"
    ]
    .dropna()
    .duplicated(keep=False)
)

if duplicate_geography_codes.any():

    duplicates = (
        icb_context.loc[
            icb_context[
                "ICB_Geography_Code"
            ]
            .duplicated(
                keep=False
            ),
            [
                "ICB_Geography_Code",
                "ICB_Code",
                "ICB_Name"
            ]
        ]
    )

    raise ValueError(
        "Duplicate ICB geography codes were found:\n"
        f"{duplicates}"
    )


# Display checks

print(
    f"ICB contextual dataset rows: "
    f"{len(icb_context)}"
)

print(
    f"ICBs with geography codes: "
    f"{icb_context['ICB_Geography_Code'].notna().sum()}"
)

print(
    f"Unique geography codes: "
    f"{icb_context['ICB_Geography_Code'].nunique()}"
)

print(
    f"ICBs with Q-codes: "
    f"{icb_context['ICB_Code'].notna().sum()}"
)

print(
    f"ICBs with population data: "
    f"{icb_context['Population_2024'].notna().sum()}"
)

print(
    f"ICBs with IMD data: "
    f"{icb_context['IMD_Average_Score'].notna().sum()}"
)

display(
    icb_context.head()
)


# Export the standalone contextual dataset

icb_context.to_excel(
    OUTPUT_DIR / "ICB_Context.xlsx",
    index=False
)

ICB contextual dataset rows: 42
ICBs with geography codes: 42
Unique geography codes: 42
ICBs with Q-codes: 42
ICBs with population data: 42
ICBs with IMD data: 42


,ICB_Geography_Code,ICB_Code,ICB_Name,Population_2024,IMD_Average_Score
0,E54000040,QOX,"Bath And North East Somerset, Swindon And Wilt...",974918.0,14.281
1,E54000024,QHG,"Bedfordshire, Luton And Milton Keynes ICB",1063247.0,19.537
2,E54000055,QHL,Birmingham And Solihull ICB,1404860.0,34.770
3,E54000062,QUA,Black Country ICB,1262719.0,30.214
4,E54000039,QUY,"Bristol, North Somerset And South Gloucestersh...",1025309.0,17.706


In [16]:
coverage_summary = create_coverage_summary(
    datasets=indicator_datasets,
    config=active_config,
    icb_lookup=icb_lookup
)

coverage_summary.to_excel(
    OUTPUT_DIR / "Coverage_Summary.xlsx",
    index=False
)

In [17]:
# to consider unexpected ICB codes as well as those missing

def create_icb_code_report(
    datasets,
    config,
    icb_lookup
):
    records = []

    expected_codes = set(icb_lookup["ICB_Code"])

    for indicator_name, df in datasets.items():
        available_codes = set(df["ICB_Code"])

        missing_codes = expected_codes - available_codes
        unexpected_codes = available_codes - expected_codes

        config_row = config.loc[
            config["Indicator"].eq(indicator_name)
        ].iloc[0]

        for code in sorted(missing_codes):
            icb_name = icb_lookup.loc[
                icb_lookup["ICB_Code"].eq(code),
                "ICB_Name"
            ].iloc[0]

            records.append({
                "Specialty": config_row["Specialty"],
                "Domain": config_row["Domain"],
                "Indicator": indicator_name,
                "Issue": "Missing expected code",
                "ICB_Code": code,
                "ICB_Name": icb_name
            })

        for code in sorted(unexpected_codes):
            source_names = (
                df.loc[
                    df["ICB_Code"].eq(code),
                    "ICB_Name"
                ]
                .dropna()
                .unique()
                .tolist()
            )

            records.append({
                "Specialty": config_row["Specialty"],
                "Domain": config_row["Domain"],
                "Indicator": indicator_name,
                "Issue": "Unexpected source code",
                "ICB_Code": code,
                "ICB_Name": "; ".join(source_names)
            })

    columns = [
        "Specialty",
        "Domain",
        "Indicator",
        "Issue",
        "ICB_Code",
        "ICB_Name"
    ]

    return pd.DataFrame(
        records,
        columns=columns
    )

In [18]:
icb_code_report = create_icb_code_report(
    datasets=indicator_datasets,
    config=active_config,
    icb_lookup=icb_lookup
)

display(icb_code_report)

icb_code_report.to_excel(
    OUTPUT_DIR / "ICB_Code_Report.xlsx",
    index=False
)

,Specialty,Domain,Indicator,Issue,ICB_Code,ICB_Name
0,Respiratory,Outcome,Under75_Respiratory_Mortality,Missing expected code,QMJ,North Central London ICB
1,Respiratory,Outcome,Under75_Respiratory_Mortality,Missing expected code,QRV,North West London ICB
2,Respiratory,Pressure,Respiratory_ElectiveAdmissions_20day+,Missing expected code,QHG,"Bedfordshire, Luton And Milton Keynes ICB"


## Build Specialty Master Datasets

In [19]:
def build_specialty_master(
    specialty,
    config,
    datasets,
    icb_lookup
):
    specialty_indicators = (
        config.loc[
            config["Specialty"].eq(specialty),
            "Indicator"
        ]
        .tolist()
    )

    specialty_master = icb_lookup.copy()

    for indicator_name in specialty_indicators:
        if indicator_name not in datasets:
            print(
                f"Skipping {indicator_name}: "
                f"dataset was not loaded"
            )
            continue

        indicator_df = datasets[indicator_name][
            ["ICB_Code", indicator_name]
        ].copy()

        specialty_master = specialty_master.merge(
            indicator_df,
            on="ICB_Code",
            how="left",
            validate="one_to_one"
        )

    return specialty_master

In [20]:
specialty_masters = {}

specialties = sorted(
    active_config["Specialty"].unique()
)

for specialty in specialties:
    specialty_master = build_specialty_master(
        specialty=specialty,
        config=active_config,
        datasets=indicator_datasets,
        icb_lookup=icb_lookup
    )

    specialty_masters[specialty] = specialty_master

    safe_specialty_name = (
        specialty
        .replace(" ", "_")
        .replace("/", "_")
    )

    specialty_master.to_excel(
        OUTPUT_DIR / f"{safe_specialty_name}_Master.xlsx",
        index=False
    )

    print(
        f"{specialty} master saved: "
        f"{len(specialty_master)} rows"
    )

Cardiovascular master saved: 42 rows
Respiratory master saved: 42 rows


## Build Combined Datasets

In [21]:
specialty_master = icb_lookup.copy()

for specialty, specialty_df in specialty_masters.items():
    specialty_indicator_columns = [
        column for column in specialty_df.columns
        if column not in ["ICB_Code", "ICB_Name"]
    ]

    specialty_master = specialty_master.merge(
        specialty_df[
            ["ICB_Code"] + specialty_indicator_columns
        ],
        on="ICB_Code",
        how="left",
        validate="one_to_one"
    )

# Add population, deprivation and mapping context

specialty_master = specialty_master.merge(
    icb_context[
        [
            "ICB_Geography_Code",
            "ICB_Code",
            "Population_2024",
            "IMD_Average_Score"
        ]
    ],
    on="ICB_Code",
    how="left",
    validate="one_to_one"
)

clinical_indicator_columns = [
    indicator
    for indicator in active_config["Indicator"]
    if indicator in specialty_master.columns
]

specialty_master = specialty_master[
    [
        "ICB_Geography_Code",
        "ICB_Code",
        "ICB_Name",
        "Population_2024",
        "IMD_Average_Score"
    ]
    + clinical_indicator_columns
]

specialty_master.to_excel(
    OUTPUT_DIR / "Specialty_Master.xlsx",
    index=False
)

print("Rows:", len(specialty_master))
print(
    "Unique ICB codes:",
    specialty_master["ICB_Code"].nunique()
)

Rows: 42
Unique ICB codes: 42


## Exploratory Data Analysis

In [22]:
clinical_indicator_columns = [
    indicator
    for indicator in active_config["Indicator"]
    if indicator in specialty_master.columns
]

context_columns = [
    column
    for column in [
        "Population_2024",
        "IMD_Average_Score"
    ]
    if column in specialty_master.columns
]

identifier_columns = [
    "ICB_Geography_Code",
    "ICB_Code",
    "ICB_Name"
]

print(
    f"Clinical indicators included in EDA: "
    f"{len(clinical_indicator_columns)}"
)

print(
    f"Contextual variables available: "
    f"{len(context_columns)}"
)

print(
    "Clinical indicators:",
    clinical_indicator_columns
)

print(
    "Contextual variables:",
    context_columns
)


Clinical indicators included in EDA: 14
Contextual variables available: 2
Clinical indicators: ['Under75_CVD_Mortality', 'Cardiology_RTT_18plus', 'Cardiology_ElectiveAdmissions_20day+', 'Cardiology_EmergencyAdmissions_20day+', 'Heart_Failure_Prevalence', 'Hypertension_Prevalence', 'CoronaryHeartDisease_Prevalence', 'PeripheralArterialDisease_Prevalence', 'Under75_Respiratory_Mortality', 'Respiratory_RTT_18plus', 'Respiratory_ElectiveAdmissions_20day+', 'Respiratory_EmergencyAdmissions_20day+', 'COPD_Prevalence', 'Asthma_Prevalence']
Contextual variables: ['Population_2024', 'IMD_Average_Score']


### EDA1: Indicator Summary Stats

In [23]:
indicator_summary = (
    specialty_master[
        clinical_indicator_columns
    ]
    .describe()
    .T
)

indicator_summary["Median"] = (
    specialty_master[
        clinical_indicator_columns
    ]
    .median()
)

indicator_summary["Missing"] = (
    specialty_master[
        clinical_indicator_columns
    ]
    .isna()
    .sum()
)

indicator_summary["Missing_Percent"] = (
    specialty_master[
        clinical_indicator_columns
    ]
    .isna()
    .mean()
    .mul(100)
)

indicator_summary["Range"] = (
    indicator_summary["max"]
    - indicator_summary["min"]
)

indicator_summary[
    "Coefficient_of_Variation"
] = (
    indicator_summary["std"]
    .div(
        indicator_summary["mean"].abs()
    )
    .replace(
        [np.inf, -np.inf],
        np.nan
    )
)

indicator_summary = (
    indicator_summary
    .reset_index()
    .rename(
        columns={
            "index": "Indicator"
        }
    )
    .merge(
        active_config[
            [
                "Indicator",
                "Specialty",
                "Domain",
                "Direction",
                "Source"
            ]
        ],
        on="Indicator",
        how="left",
        validate="one_to_one"
    )
)

display(
    indicator_summary.round(3)
)

indicator_summary.to_excel(
    OUTPUT_DIR / "Indicator_Summary.xlsx",
    index=False
)

,Indicator,count,mean,std,min,25%,50%,75%,max,Median,Missing,Missing_Percent,Range,Coefficient_of_Variation,Specialty,Domain,Direction,Source
0,Under75_CVD_Mortality,42.0,69.660,11.499,52.70,60.100,65.600,78.700,96.70,65.600,0,0.000,44.00,0.165,Cardiovascular,Outcome,Higher_Worse,MHS
1,Cardiology_RTT_18plus,42.0,37.521,8.927,16.74,30.848,38.600,44.732,53.03,38.600,0,0.000,36.29,0.238,Cardiovascular,Pressure,Higher_Worse,MHS
2,Cardiology_ElectiveAdmissions_20day+,42.0,3.050,3.525,0.00,0.700,2.150,3.900,14.30,2.150,0,0.000,14.30,1.156,Cardiovascular,Pressure,Higher_Worse,MHS
3,Cardiology_EmergencyAdmissions_20day+,42.0,5.729,2.837,0.00,3.900,5.550,7.250,13.50,5.550,0,0.000,13.50,0.495,Cardiovascular,Pressure,Higher_Worse,MHS
4,Heart_Failure_Prevalence,42.0,1.148,0.296,0.60,1.000,1.100,1.375,1.80,1.100,0,0.000,1.20,0.258,Cardiovascular,Burden,Higher_Worse,MHS
5,Hypertension_Prevalence,42.0,15.633,2.210,10.90,14.300,16.000,17.400,19.00,16.000,0,0.000,8.10,0.141,Cardiovascular,Burden,Higher_Worse,MHS
6,CoronaryHeartDisease_Prevalence,42.0,3.086,0.687,1.80,2.700,3.050,3.500,4.60,3.050,0,0.000,2.80,0.223,Cardiovascular,Burden,Higher_Worse,MHS
7,PeripheralArterialDisease_Prevalence,42.0,0.552,0.153,0.30,0.400,0.500,0.700,0.80,0.500,0,0.000,0.50,0.278,Cardiovascular,Burden,Higher_Worse,MHS
8,Under75_Respiratory_Mortality,40.0,19.525,5.679,2.00,16.750,20.000,23.000,31.00,20.000,2,4.762,29.00,0.291,Respiratory,Outcome,Higher_Worse,MHS
9,Respiratory_RTT_18plus,42.0,35.012,10.110,13.14,27.710,36.555,41.755,59.35,36.555,0,0.000,46.21,0.289,Respiratory,Pressure,Higher_Worse,MHS


### EDA2: High Variation Indicators
- Identifies specialites with unequal performance and indicators with biggest geographic variation

In [24]:
high_variation = (
    indicator_summary
    .sort_values(
        "Coefficient_of_Variation",
        ascending=False
    )
)

display(
    high_variation[
        [
            "Specialty",
            "Domain",
            "Indicator",
            "Coefficient_of_Variation"
        ]
    ]
)

,Specialty,Domain,Indicator,Coefficient_of_Variation
10,Respiratory,Pressure,Respiratory_ElectiveAdmissions_20day+,1.721265
2,Cardiovascular,Pressure,Cardiology_ElectiveAdmissions_20day+,1.155777
11,Respiratory,Pressure,Respiratory_EmergencyAdmissions_20day+,0.501546
3,Cardiovascular,Pressure,Cardiology_EmergencyAdmissions_20day+,0.495323
8,Respiratory,Outcome,Under75_Respiratory_Mortality,0.290879
9,Respiratory,Pressure,Respiratory_RTT_18plus,0.288751
7,Cardiovascular,Burden,PeripheralArterialDisease_Prevalence,0.277702
12,Respiratory,Burden,COPD_Prevalence,0.261698
4,Cardiovascular,Burden,Heart_Failure_Prevalence,0.257656
1,Cardiovascular,Pressure,Cardiology_RTT_18plus,0.237917


### EDA3: Top and Bottom 5 ICBs
- Helpful for spotting obvious outliers and/or coding issues

In [25]:
for indicator in clinical_indicator_columns:

    print("\n")
    print("="*60)
    print(indicator)
    print("="*60)

    print("\nHighest")

    display(
        specialty_master[
            ["ICB_Name", indicator]
        ]
        .sort_values(
            indicator,
            ascending=False
        )
        .head(5)
    )

    print("\nLowest")

    display(
        specialty_master[
            ["ICB_Name", indicator]
        ]
        .sort_values(
            indicator
        )
        .head(5)
    )



Under75_CVD_Mortality

Highest


,ICB_Name,Under75_CVD_Mortality
3,Black Country ICB,96.7
15,Greater Manchester ICB,93.9
21,Lancashire And South Cumbria ICB,89.4
36,South Yorkshire ICB,85.9
31,Nottingham And Nottinghamshire ICB,84.2



Lowest


,ICB_Name,Under75_CVD_Mortality
6,Cambridgeshire And Peterborough ICB,52.7
5,"Buckinghamshire, Oxfordshire And Berkshire Wes...",54.3
18,Hertfordshire And West Essex ICB,56.9
35,South West London ICB,57.0
13,Frimley ICB,58.2




Cardiology_RTT_18plus

Highest


...

,ICB_Name,COPD_Prevalence
29,North West London ICB,0.9
26,North Central London ICB,1.0
28,North East London ICB,1.0
35,South West London ICB,1.1
34,South East London ICB,1.2




Asthma_Prevalence

Highest


,ICB_Name,Asthma_Prevalence
25,Norfolk And Waveney ICB,8.1
8,Cornwall And The Isles Of Scilly ICB,8.0
11,Devon ICB,7.8
12,Dorset ICB,7.8
21,Lancashire And South Cumbria ICB,7.6



Lowest


,ICB_Name,Asthma_Prevalence
29,North West London ICB,4.4
26,North Central London ICB,4.5
28,North East London ICB,4.8
35,South West London ICB,5.0
34,South East London ICB,5.0


### EDA4: Correlation Matrix
- these are descriptive correlations across ICBs - they do not show that one indicator causes another but they may highlight related signals or indicator overlap

In [26]:
## EDA 4: Correlation Analysis

# Create a reliable list of indicators from the configuration
# rather than reusing indicator_columns from an earlier loop

all_indicator_columns = [
    indicator
    for indicator in active_config["Indicator"]
    if indicator in specialty_master.columns
]

print(
    f"Number of indicators included in "
    f"correlation analysis: {len(all_indicator_columns)}"
)


# ---------------------------------------------------------
# 1. Correlation matrices for each specialty
# ---------------------------------------------------------

correlation_matrices_by_specialty = {}

for specialty in sorted(
    active_config["Specialty"].unique()
):

    specialty_indicators = (
        active_config.loc[
            active_config["Specialty"].eq(specialty),
            "Indicator"
        ]
        .tolist()
    )

    # Only retain indicators that successfully loaded
    # and are present in the combined master dataset

    available_indicators = [
        indicator
        for indicator in specialty_indicators
        if indicator in specialty_master.columns
    ]

    if len(available_indicators) < 2:
        print(
            f"Skipping {specialty}: fewer than "
            f"two available indicators"
        )
        continue

    specialty_correlation = (
        specialty_master[
            available_indicators
        ]
        .corr(
            method="pearson",
            min_periods=10
        )
    )

    correlation_matrices_by_specialty[
        specialty
    ] = specialty_correlation

    print("\n" + "=" * 70)
    print(f"{specialty}: correlation matrix")
    print("=" * 70)

    display(
        specialty_correlation.round(2)
    )


# Export every specialty matrix to a separate worksheet

with pd.ExcelWriter(
    OUTPUT_DIR / "Indicator_Correlations_By_Specialty.xlsx",
    engine="openpyxl"
) as writer:

    for specialty, matrix in (
        correlation_matrices_by_specialty.items()
    ):

        safe_sheet_name = (
            specialty
            .replace("/", "_")
        )[:31]

        matrix.to_excel(
            writer,
            sheet_name=safe_sheet_name
        )


# ---------------------------------------------------------
# 2. Combined correlation matrix across all specialties
# ---------------------------------------------------------

combined_correlation_matrix = (
    specialty_master[
        all_indicator_columns
    ]
    .corr(
        method="pearson",
        min_periods=10
    )
)

print("\n" + "=" * 70)
print("Combined correlation matrix: all indicators")
print("=" * 70)

display(
    combined_correlation_matrix.round(2)
)

combined_correlation_matrix.to_excel(
    OUTPUT_DIR / "Indicator_Correlations_All.xlsx"
)


# ---------------------------------------------------------
# 3. Extract the strongest correlations
# ---------------------------------------------------------

def extract_correlation_pairs(
    correlation_matrix,
    minimum_absolute_correlation=0.5
):

    # Keep only the upper triangle so that each
    # indicator pair appears once

    upper_triangle = correlation_matrix.where(
        np.triu(
            np.ones(
                correlation_matrix.shape,
                dtype=bool
            ),
            k=1
        )
    )

    correlation_pairs = (
        upper_triangle
        .stack()
        .reset_index()
    )

    correlation_pairs.columns = [
        "Indicator_1",
        "Indicator_2",
        "Correlation"
    ]

    correlation_pairs[
        "Absolute_Correlation"
    ] = (
        correlation_pairs["Correlation"]
        .abs()
    )

    correlation_pairs = (
        correlation_pairs.loc[
            correlation_pairs[
                "Absolute_Correlation"
            ] >= minimum_absolute_correlation
        ]
        .sort_values(
            "Absolute_Correlation",
            ascending=False
        )
        .reset_index(drop=True)
    )

    return correlation_pairs


strong_correlations = extract_correlation_pairs(
    correlation_matrix=combined_correlation_matrix,
    minimum_absolute_correlation=0.5
)

print("\n" + "=" * 70)
print("Strong indicator correlations")
print("=" * 70)

display(
    strong_correlations.round(2)
)

strong_correlations.to_excel(
    OUTPUT_DIR / "Strong_Indicator_Correlations.xlsx",
    index=False
)

Number of indicators included in correlation analysis: 14

Cardiovascular: correlation matrix


,Under75_CVD_Mortality,Cardiology_RTT_18plus,Cardiology_ElectiveAdmissions_20day+,Cardiology_EmergencyAdmissions_20day+,Heart_Failure_Prevalence,Hypertension_Prevalence,CoronaryHeartDisease_Prevalence,PeripheralArterialDisease_Prevalence
Under75_CVD_Mortality,1.00,-0.02,-0.08,-0.15,0.08,0.10,0.14,0.31
Cardiology_RTT_18plus,-0.02,1.00,-0.01,0.17,-0.19,-0.13,-0.15,-0.08
Cardiology_ElectiveAdmissions_20day+,-0.08,-0.01,1.00,0.23,0.09,0.17,0.07,0.12
Cardiology_EmergencyAdmissions_20day+,-0.15,0.17,0.23,1.00,-0.17,-0.25,-0.24,-0.20
Heart_Failure_Prevalence,0.08,-0.19,0.09,-0.17,1.00,0.88,0.91,0.80
Hypertension_Prevalence,0.10,-0.13,0.17,-0.25,0.88,1.00,0.92,0.82
CoronaryHeartDisease_Prevalence,0.14,-0.15,0.07,-0.24,0.91,0.92,1.00,0.85
PeripheralArterialDisease_Prevalence,0.31,-0.08,0.12,-0.20,0.80,0.82,0.85,1.00



Respiratory: correlation matrix


,Under75_Respiratory_Mortality,Respiratory_RTT_18plus,Respiratory_ElectiveAdmissions_20day+,Respiratory_EmergencyAdmissions_20day+,COPD_Prevalence,Asthma_Prevalence
Under75_Respiratory_Mortality,1.00,-0.07,0.10,0.11,0.84,0.66
Respiratory_RTT_18plus,-0.07,1.00,0.04,-0.02,-0.21,-0.10
Respiratory_ElectiveAdmissions_20day+,0.10,0.04,1.00,0.26,0.15,0.14
Respiratory_EmergencyAdmissions_20day+,0.11,-0.02,0.26,1.00,-0.09,-0.22
COPD_Prevalence,0.84,-0.21,0.15,-0.09,1.00,0.87
Asthma_Prevalence,0.66,-0.10,0.14,-0.22,0.87,1.00



Combined correlation matrix: all indicators


,Under75_CVD_Mortality,Cardiology_RTT_18plus,Cardiology_ElectiveAdmissions_20day+,Cardiology_EmergencyAdmissions_20day+,Heart_Failure_Prevalence,Hypertension_Prevalence,CoronaryHeartDisease_Prevalence,PeripheralArterialDisease_Prevalence,Under75_Respiratory_Mortality,Respiratory_RTT_18plus,Respiratory_ElectiveAdmissions_20day+,Respiratory_EmergencyAdmissions_20day+,COPD_Prevalence,Asthma_Prevalence
Under75_CVD_Mortality,1.00,-0.02,-0.08,-0.15,0.08,0.10,0.14,0.31,0.56,-0.17,0.04,0.05,0.38,0.11
Cardiology_RTT_18plus,-0.02,1.00,-0.01,0.17,-0.19,-0.13,-0.15,-0.08,-0.01,0.15,0.07,0.22,-0.14,-0.17
Cardiology_ElectiveAdmissions_20day+,-0.08,-0.01,1.00,0.23,0.09,0.17,0.07,0.12,-0.01,0.23,-0.09,-0.48,0.03,0.21
Cardiology_EmergencyAdmissions_20day+,-0.15,0.17,0.23,1.00,-0.17,-0.25,-0.24,-0.20,-0.03,0.16,0.05,-0.01,-0.11,-0.13
Heart_Failure_Prevalence,0.08,-0.19,0.09,-0.17,1.00,0.88,0.91,0.80,0.52,-0.20,0.22,-0.22,0.80,0.85
Hypertension_Prevalence,0.10,-0.13,0.17,-0.25,0.88,1.00,0.92,0.82,0.64,-0.05,0.15,-0.18,0.85,0.90
CoronaryHeartDisease_Prevalence,0.14,-0.15,0.07,-0.24,0.91,0.92,1.00,0.85,0.63,-0.09,0.08,-0.22,0.88,0.91
PeripheralArterialDisease_Prevalence,0.31,-0.08,0.12,-0.20,0.80,0.82,0.85,1.00,0.74,-0.16,0.11,-0.16,0.89,0.81
Under75_Respiratory_Mortality,0.56,-0.01,-0.01,-0.03,0.52,0.64,0.63,0.74,1.00,-0.07,0.10,0.11,0.84,0.66
Respiratory_RTT_18plus,-0.17,0.15,0.23,0.16,-0.20,-0.05,-0.09,-0.16,-0.07,1.00,0.04,-0.02,-0.21,-0.10



Strong indicator correlations


,Indicator_1,Indicator_2,Correlation,Absolute_Correlation
0,Hypertension_Prevalence,CoronaryHeartDisease_Prevalence,0.92,0.92
1,CoronaryHeartDisease_Prevalence,Asthma_Prevalence,0.91,0.91
2,Heart_Failure_Prevalence,CoronaryHeartDisease_Prevalence,0.91,0.91
3,Hypertension_Prevalence,Asthma_Prevalence,0.90,0.90
4,PeripheralArterialDisease_Prevalence,COPD_Prevalence,0.89,0.89
5,CoronaryHeartDisease_Prevalence,COPD_Prevalence,0.88,0.88
6,Heart_Failure_Prevalence,Hypertension_Prevalence,0.88,0.88
7,COPD_Prevalence,Asthma_Prevalence,0.87,0.87
8,Heart_Failure_Prevalence,Asthma_Prevalence,0.85,0.85
9,Hypertension_Prevalence,COPD_Prevalence,0.85,0.85


### EDA5: Population and Inequalities Exploration

In [27]:
context_columns = [
    "Population_2024",
    "IMD_Average_Score"
]

clinical_context_correlations = (
    specialty_master[
        clinical_indicator_columns
        + context_columns
    ]
    .corr(
        method="pearson",
        min_periods=10
    )
    .loc[
        clinical_indicator_columns,
        context_columns
    ]
)

print("\n" + "=" * 70)
print(
    "Clinical indicator associations with "
    "population and deprivation"
)
print("=" * 70)

display(
    clinical_context_correlations.round(2)
)

clinical_context_correlations.to_excel(
    OUTPUT_DIR
    / "Clinical_Context_Correlations.xlsx"
)


Clinical indicator associations with population and deprivation


,Population_2024,IMD_Average_Score
Under75_CVD_Mortality,0.39,0.82
Cardiology_RTT_18plus,-0.01,-0.09
Cardiology_ElectiveAdmissions_20day+,-0.27,-0.10
Cardiology_EmergencyAdmissions_20day+,0.15,0.02
Heart_Failure_Prevalence,-0.30,-0.01
Hypertension_Prevalence,-0.29,-0.04
CoronaryHeartDisease_Prevalence,-0.22,0.09
PeripheralArterialDisease_Prevalence,0.06,0.25
Under75_Respiratory_Mortality,0.28,0.58
Respiratory_RTT_18plus,-0.11,0.04


In [28]:
imd_associations = (
    clinical_context_correlations[
        ["IMD_Average_Score"]
    ]
    .reset_index()
    .rename(
        columns={
            "index": "Indicator",
            "IMD_Average_Score":
                "Correlation_With_IMD"
        }
    )
)

imd_associations = (
    imd_associations
    .merge(
        active_config[
            [
                "Indicator",
                "Specialty",
                "Domain"
            ]
        ],
        on="Indicator",
        how="left"
    )
    .sort_values(
        "Correlation_With_IMD",
        ascending=False
    )
)

display(
    imd_associations.round(2)
)

imd_associations.to_excel(
    OUTPUT_DIR / "IMD_Associations.xlsx",
    index=False
)

,Indicator,Correlation_With_IMD,Specialty,Domain
0,Under75_CVD_Mortality,0.82,Cardiovascular,Outcome
8,Under75_Respiratory_Mortality,0.58,Respiratory,Outcome
12,COPD_Prevalence,0.31,Respiratory,Burden
7,PeripheralArterialDisease_Prevalence,0.25,Cardiovascular,Burden
11,Respiratory_EmergencyAdmissions_20day+,0.17,Respiratory,Pressure
6,CoronaryHeartDisease_Prevalence,0.09,Cardiovascular,Burden
9,Respiratory_RTT_18plus,0.04,Respiratory,Pressure
10,Respiratory_ElectiveAdmissions_20day+,0.03,Respiratory,Pressure
13,Asthma_Prevalence,0.03,Respiratory,Burden
3,Cardiology_EmergencyAdmissions_20day+,0.02,Cardiovascular,Pressure


### EDA6: Population Impact

In [29]:
population_impact = []

for indicator in clinical_indicator_columns:

    indicator_data = specialty_master[
        [
            "ICB_Code",
            "Population_2024",
            indicator
        ]
    ].dropna()

    population_correlation = (
        indicator_data[
            ["Population_2024", indicator]
        ]
        .corr()
        .iloc[0,1]
    )

    population_impact.append({
        "Indicator": indicator,
        "Population_Correlation":
            population_correlation
    })

population_impact = pd.DataFrame(
    population_impact
)

population_impact = (
    population_impact
    .merge(
        active_config[
            [
                "Indicator",
                "Specialty",
                "Domain"
            ]
        ],
        on="Indicator",
        how="left"
    )
    .sort_values(
        "Population_Correlation",
        ascending=False
    )
)

display(
    population_impact.round(2)
)

population_impact.to_excel(
    OUTPUT_DIR / "Population_Impact.xlsx",
    index=False
)

,Indicator,Population_Correlation,Specialty,Domain
0,Under75_CVD_Mortality,0.39,Cardiovascular,Outcome
8,Under75_Respiratory_Mortality,0.28,Respiratory,Outcome
11,Respiratory_EmergencyAdmissions_20day+,0.27,Respiratory,Pressure
3,Cardiology_EmergencyAdmissions_20day+,0.15,Cardiovascular,Pressure
7,PeripheralArterialDisease_Prevalence,0.06,Cardiovascular,Burden
12,COPD_Prevalence,0.04,Respiratory,Burden
10,Respiratory_ElectiveAdmissions_20day+,0.01,Respiratory,Pressure
1,Cardiology_RTT_18plus,-0.01,Cardiovascular,Pressure
9,Respiratory_RTT_18plus,-0.11,Respiratory,Pressure
6,CoronaryHeartDisease_Prevalence,-0.22,Cardiovascular,Burden


### EDA7: Specialty Signal Profiles

In [30]:
# create specialty signal profiles

specialty_profiles = (
    indicator_summary
    .groupby(
        [
            "Specialty",
            "Domain"
        ],
        as_index=False
    )
    .agg(
        Mean_CV=(
            "Coefficient_of_Variation",
            "mean"
        )
    )
)

In [31]:
# bring in IMD signal

imd_profile = (
    imd_associations
    .groupby(
        "Specialty",
        as_index=False
    )
    .agg(
        Mean_IMD_Association=(
            "Correlation_With_IMD",
            "mean"
        )
    )
)

In [32]:
# pivot domains to columns

specialty_profile_table = (
    specialty_profiles
    .pivot(
        index="Specialty",
        columns="Domain",
        values="Mean_CV"
    )
    .reset_index()
)

In [33]:
# add inequalities

specialty_profile_table = (
    specialty_profile_table
    .merge(
        imd_profile,
        on="Specialty",
        how="left"
    )
)

In [34]:
display(
    specialty_profile_table.round(3)
)

,Specialty,Burden,Outcome,Pressure,Mean_IMD_Association
0,Cardiovascular,0.225,0.165,0.630,0.118
1,Respiratory,0.200,0.291,0.837,0.194


In [35]:
#convert numbers into labels

def signal_label(series):

    low = series.quantile(0.33)
    high = series.quantile(0.67)

    return pd.cut(
        series,
        bins=[
            -np.inf,
            low,
            high,
            np.inf
        ],
        labels=[
            "Low",
            "Medium",
            "High"
        ]
    )

In [36]:
for column in [
    "Burden",
    "Outcome",
    "Pressure",
    "Mean_IMD_Association"
]:

    specialty_profile_table[
        column + "_Profile"
    ] = signal_label(
        specialty_profile_table[column]
    )

In [37]:
specialty_profile_display = (
    specialty_profile_table[
        [
            "Specialty",
            "Burden_Profile",
            "Outcome_Profile",
            "Pressure_Profile",
            "Mean_IMD_Association_Profile"
        ]
    ]
    .rename(
        columns={
            "Burden_Profile":
                "Burden",
            "Outcome_Profile":
                "Outcomes",
            "Pressure_Profile":
                "Pressure",
            "Mean_IMD_Association_Profile":
                "IMD_Association"
        }
    )
)

display(
    specialty_profile_display
)

specialty_profile_display.to_excel(
    OUTPUT_DIR
    / "Specialty_Signal_Profiles.xlsx",
    index=False
)

,Specialty,Burden,Outcomes,Pressure,IMD_Association
0,Cardiovascular,High,Low,Low,Low
1,Respiratory,Low,High,High,High


In [38]:
with pd.ExcelWriter(
    OUTPUT_DIR
    / "Specialty_Signal_Profiles.xlsx",
    engine="openpyxl"
) as writer:

    specialty_profile_display.to_excel(
        writer,
        sheet_name="Dashboard_Profile",
        index=False
    )

    specialty_profile_table.to_excel(
        writer,
        sheet_name="Detailed_Profile",
        index=False
    )

In [39]:
print(specialty_profile_table.columns.tolist())

['Specialty', 'Burden', 'Outcome', 'Pressure', 'Mean_IMD_Association', 'Burden_Profile', 'Outcome_Profile', 'Pressure_Profile', 'Mean_IMD_Association_Profile']


In [40]:
### Indicator Dictionary

indicator_dictionary = (
    active_config[
        [
            "Specialty",
            "Domain",
            "Indicator",
            "Source",
            "Direction",
            "File_Name"
        ]
    ]
    .copy()
)

indicator_dictionary["Unit"] = np.nan
indicator_dictionary["Population_Adjusted"] = np.nan
indicator_dictionary["Population_Conversion_Valid"] = np.nan
indicator_dictionary["Need_Dimension"] = np.nan
indicator_dictionary["Benchmark_Available"] = np.nan
indicator_dictionary["Notes"] = np.nan

display(
    indicator_dictionary
)

indicator_dictionary.to_excel(
    OUTPUT_DIR / "Indicator_Dictionary.xlsx",
    index=False
)

,Specialty,Domain,Indicator,Source,Direction,File_Name,Unit,Population_Adjusted,Population_Conversion_Valid,Need_Dimension,Benchmark_Available,Notes
0,Cardiovascular,Outcome,Under75_CVD_Mortality,MHS,Higher_Worse,Under75_CVD_Mortality.xlsx,NaN,NaN,NaN,NaN,NaN,NaN
1,Cardiovascular,Pressure,Cardiology_RTT_18plus,MHS,Higher_Worse,Cardiology_RTT_18plus.xlsx,NaN,NaN,NaN,NaN,NaN,NaN
2,Cardiovascular,Pressure,Cardiology_ElectiveAdmissions_20day+,MHS,Higher_Worse,Cardiology_ElectiveAdmissions_20day+.xlsx,NaN,NaN,NaN,NaN,NaN,NaN
3,Cardiovascular,Pressure,Cardiology_EmergencyAdmissions_20day+,MHS,Higher_Worse,Cardiology_EmergencyAdmissions_20day+.xlsx,NaN,NaN,NaN,NaN,NaN,NaN
4,Cardiovascular,Burden,Heart_Failure_Prevalence,MHS,Higher_Worse,Heart_Failure_Prevalence.xlsx,NaN,NaN,NaN,NaN,NaN,NaN
5,Cardiovascular,Burden,Hypertension_Prevalence,MHS,Higher_Worse,Hypertension_Prevalence.xlsx,NaN,NaN,NaN,NaN,NaN,NaN
6,Cardiovascular,Burden,CoronaryHeartDisease_Prevalence,MHS,Higher_Worse,CoronaryHeartDisease_Prevalence.xlsx,NaN,NaN,NaN,NaN,NaN,NaN
7,Cardiovascular,Burden,PeripheralArterialDisease_Prevalence,MHS,Higher_Worse,PeripheralArterialDisease_Prevalence.xlsx,NaN,NaN,NaN,NaN,NaN,NaN
8,Respiratory,Outcome,Under75_Respiratory_Mortality,MHS,Higher_Worse,Under75_Respiratory_Mortality.xlsx,NaN,NaN,NaN,NaN,NaN,NaN
9,Respiratory,Pressure,Respiratory_RTT_18plus,MHS,Higher_Worse,Respiratory_RTT_18plus.xlsx,NaN,NaN,NaN,NaN,NaN,NaN


## RUN SUMMARY

In [41]:
print("=" * 50)
print("RUN SUMMARY")
print("=" * 50)

print(
    f"Specialties analysed: "
    f"{active_config['Specialty'].nunique()}"
)

print(
    f"Indicators analysed: "
    f"{len(indicator_datasets)}"
)

print(
    f"ICB geography codes available: "
    f"{specialty_master['ICB_Geography_Code'].notna().sum()}"
)

print(
    f"Population records available: "
    f"{specialty_master['Population_2024'].notna().sum()}"
)

print(
    f"IMD records available: "
    f"{specialty_master['IMD_Average_Score'].notna().sum()}"
)

RUN SUMMARY
Specialties analysed: 2
Indicators analysed: 14
ICB geography codes available: 42
Population records available: 42
IMD records available: 42


## Signal Framework
- 